# SFT Data Scaling & Empirical Learning Curve Analysis

Dieses Notebook analysiert systematisch das **Skalierungsverhalten des Supervised Fine-Tuning (SFT)** Modells (`facebook/mbart-large-50` + LoRA) entlang der verfügbaren Trainingsdatenmenge:

$$\text{Trainingsfraktionen } F \in \{10\%, 25\%, 50\%, 75\%, 100\%\} \iff N \in \{65, 162, 323, 484, 646\} \text{ Artikelpaare}$$

### Zentrale Forschungsfragen:
1. **Neural Scaling Law & Cross-Entropy Loss:** Folgt der Validierungsverlust einem Potenzgesetz ($L(N) \propto N^{-\alpha}$) über die Datenfraktionen?
2. **Sample Complexity & Phasenübergang:** Bei wie vielen parallelen Trainingspaaren überwindet mBART die Identitäts-Kopierfalle und beginnt mit der Generierung echter Leichte-Sprache-Strukturen (SVO, Bindestrich-Zerlegung)?
3. **Plateau vs. DPO-Headroom:** Wo setzt die Sättigung der sprachlichen Einfachheit ($R_{\text{style}}$) im SFT-Paradigma ein und wie viel Mehrwert liefert DPO gegenüber der vollen SFT-Datenbasis?

In [ ]:
import os
import sys
import json
import glob
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import torch

def find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'data')) and os.path.exists(os.path.join(p, 'results')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.expanduser('~/Documents/Master Thesis'))

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print('Arbeitsverzeichnis:', os.getcwd())
print(f'Nutze Device: {DEVICE}')

RESULTS_DIR = os.path.join(REPO_ROOT, 'results/experiments/sft_scaling')
PLOTS_DIR = os.path.join(REPO_ROOT, 'results/plots/sft_scaling')
os.makedirs(PLOTS_DIR, exist_ok=True)
SUMMARY_CSV = os.path.join(RESULTS_DIR, 'sft_scaling_summary.csv')
if not os.path.exists(SUMMARY_CSV):
    SUMMARY_CSV = os.path.join(RESULTS_DIR, 'sft_scaling_comparison_summary.csv')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 150


## 1. Daten laden und tabellarische Übersicht

In [ ]:
if os.path.exists(SUMMARY_CSV):
    df = pd.read_csv(SUMMARY_CSV)
else:
    json_files = glob.glob(os.path.join(RESULTS_DIR, "*_metrics.json"))
    records = []
    for jf in json_files:
        try:
            with open(jf, "r", encoding="utf-8") as f:
                records.append(json.load(f))
        except Exception as e:
            print(f"Fehler beim Lesen von {jf}: {e}")
    df = pd.DataFrame(records)

if not df.empty:
    df = df.sort_values(by="train_fraction").reset_index(drop=True)
    print(f"Erfolgreich geladen: {len(df)} SFT-Skalierungsstufen")

    display_cols = [
        "experiment_name", "train_fraction", "num_train_pairs",
        "best_val_loss", "r_style_mean", "r_sem_as_mean", "sim_ref_mean",
        "composite_reward_mean", "bleu_mean", "rouge_l_mean",
        "avg_gen_tokens", "truncation_rate_pct", "training_time_seconds"
    ]
    avail_cols = [c for c in display_cols if c in df.columns]
    styled_df = df[avail_cols].copy()

    for col in styled_df.columns:
        if col not in ["experiment_name", "num_train_pairs"]:
            styled_df[col] = styled_df[col].apply(lambda x: f"{x:.4f}" if isinstance(x, (float, np.floating)) else x)

    display(styled_df)

    print()
    print("=== Markdown-Tabelle  ===")
    print()
    try:
        print(styled_df.to_markdown(index=False))
    except Exception:
        print(styled_df.to_string(index=False))
else:
    print("[HINWEIS] Noch keine Evaluationsergebnisse gefunden.")


## 2. Empirische Lern- und Skalierungskurven (6-Panel Analyse)
Visualisierung von Loss-Skalierung, sprachlicher Einfachheit, semantischer Treue, N-Gramm-Metriken, struktureller Textstabilität und Compute-Aufwand.

In [ ]:
if not df .empty :
    fig ,axes =plt .subplots (3 ,2 ,figsize =(15 ,16 ))

    x_vals =df ["num_train_pairs"]
    fractions =[f"{int (f *100 )}%"for f in df ["train_fraction"]]

    # Panel A: Cross-Entropy Loss Scaling (Validation & Train Loss)
    ax_loss =axes [0 ,0 ]
    ax_loss .plot (x_vals ,df ["best_val_loss"],marker ='o',linewidth =2.5 ,color ='#1f77b4',label ="Best Validation Loss")
    if "final_train_loss"in df .columns :
        ax_loss .plot (x_vals ,df ["final_train_loss"],marker ='s',linestyle ='--',color ='#aec7e8',label ="Final Train Loss")

        # Power-law Fit: L(N) = a * N^(-alpha) + b
    try :
        def power_law (x ,a ,alpha ,b ):
            return a *np .power (x ,-alpha )+b 
        popt ,_ =curve_fit (power_law ,x_vals ,df ["best_val_loss"],p0 =[2.0 ,0.3 ,1.5 ],maxfev =5000 )
        x_smooth =np .linspace (x_vals .min (),x_vals .max (),100 )
        ax_loss.plot(x_smooth, power_law(x_smooth, *popt), "r:", linewidth=1.8, label=fr"Fit: (N) \propto N^{{-{popt[1]:.2f}}}")
    except Exception :
        pass 

    ax_loss .set_title ("A) Cross-Entropy Loss Scaling ($L_{CE}$ vs. $N$)",fontweight ="bold")
    ax_loss .set_xlabel ("Anzahl Trainings-Artikelpaare ($N$)")
    ax_loss .set_ylabel ("Validation Loss")
    ax_loss .legend (loc ="upper right")
    for i ,(x ,y )in enumerate (zip (x_vals ,df ["best_val_loss"])):
        ax_loss .annotate (f"{y :.3f}\n({fractions [i ]})",(x ,y ),textcoords ="offset points",xytext =(0 ,7 ),ha ='center',fontsize =9 )

        # Panel B: Simplicity Score (R_style) & DPO-Headroom
    ax_simp =axes [0 ,1 ]
    ax_simp .plot (x_vals ,df ["r_style_mean"],marker ='o',linewidth =2.5 ,color ='#2ca02c',label ="SFT Simplicity ($R_{style}$)")
    ax_simp .plot (x_vals ,df ["composite_reward_mean"],marker ='s',linewidth =2 ,linestyle ='--',color ='#17becf',label ="Composite Reward")
    ax_simp .axhline (0.6938 ,color ='#d62728',linestyle ='-.',linewidth =2.0 ,label ="DPO Mean Referenz ($R_{style}=0.6938$)")
    ax_simp .set_title ("B) Sprachliche Einfachheit & DPO-Headroom",fontweight ="bold")
    ax_simp .set_xlabel ("Anzahl Trainings-Artikelpaare ($N$)")
    ax_simp .set_ylabel ("Score [0.0 - 1.0]")
    ax_simp .set_ylim (0.2 ,0.8 )
    ax_simp .legend (loc ="lower right")
    for i ,(x ,y )in enumerate (zip (x_vals ,df ["r_style_mean"])):
        ax_simp .annotate (f"{y :.3f}",(x ,y ),textcoords ="offset points",xytext =(0 ,7 ),ha ='center',fontsize =9 )

        # Panel C: Semantik (AS) & LS-Referenztreue
    ax_sem =axes [1 ,0 ]
    ax_sem .plot (x_vals ,df ["r_sem_as_mean"],marker ='^',linewidth =2.5 ,color ='#ff7f0e',label ="Semantik zu AS ($R_{sem, AS}$)")
    ax_sem .plot (x_vals ,df ["sim_ref_mean"],marker ='D',linewidth =2 ,linestyle ='-.',color ='#9467bd',label ="Treue zu LS ($Sim_{ref}$)")
    ax_sem .set_title ("C) Semantische Treue & Referenzähnlichkeit",fontweight ="bold")
    ax_sem .set_xlabel ("Anzahl Trainings-Artikelpaare ($N$)")
    ax_sem .set_ylabel ("SBERT Cosine Similarity")
    ax_sem .set_ylim (0.80 ,1.00 )
    ax_sem .legend (loc ="lower right")
    for x ,y in zip (x_vals ,df ["r_sem_as_mean"]):
        ax_sem .annotate (f"{y :.3f}",(x ,y ),textcoords ="offset points",xytext =(0 ,7 ),ha ='center',fontsize =9 )

        # Panel D: Lexikalische N-Gramm Metriken: BLEU & ROUGE-L
    ax_lex =axes [1 ,1 ]
    ax_lex .plot (x_vals ,df ["rouge_l_mean"],marker ='o',linewidth =2.5 ,color ='#e377c2',label ="ROUGE-L F1")
    ax_lex_twin =ax_lex .twinx ()
    ax_lex_twin .plot (x_vals ,df ["bleu_mean"],marker ='s',linewidth =2.0 ,linestyle =':',color ='#8c564b',label ="BLEU")
    ax_lex .set_title ("D) Lexikalische Überlappung (ROUGE-L / BLEU)",fontweight ="bold")
    ax_lex .set_xlabel ("Anzahl Trainings-Artikelpaare ($N$)")
    ax_lex .set_ylabel ("ROUGE-L F1",color ='#e377c2')
    ax_lex_twin .set_ylabel ("BLEU",color ='#8c564b')
    ax_lex .grid (False )

    # Panel E: Satzabbruchquote & Generierte Textlänge
    ax_struct =axes [2 ,0 ]
    ax_struct_twin =ax_struct .twinx ()
    ax_struct .bar ([x -12 for x in x_vals ],df ["truncation_rate_pct"],width =24 ,color ='#bcbd22',alpha =0.75 ,label ="Truncation Rate (%)")
    ax_struct_twin .plot (x_vals ,df ["avg_gen_tokens"],marker ='o',color ='#393b79',linewidth =2.5 ,label ="Ø Tokens")
    ax_struct .set_title ("E) Strukturelle Stabilität (Abbruchquote & Länge)",fontweight ="bold")
    ax_struct .set_xlabel ("Anzahl Trainings-Artikelpaare ($N$)")
    ax_struct .set_ylabel ("Truncation Rate (%)",color ='#bcbd22')
    ax_struct_twin .set_ylabel ("Ø Generierte Tokens",color ='#393b79')
    ax_struct .set_ylim (0 ,100 )
    ax_struct .grid (False )

    # Panel F: Trainingszeit & Rechenaufwand (Wall-Clock Time)
    ax_time =axes [2 ,1 ]
    ax_time .plot (x_vals ,df ["training_time_seconds"]/60.0 ,marker ='s',linewidth =2.5 ,color ='#7f7f7f',label ="Training Time (min)")
    ax_time .set_title ("F) Rechenaufwand & Trainingszeit",fontweight ="bold")
    ax_time .set_xlabel ("Anzahl Trainings-Artikelpaare ($N$)")
    ax_time .set_ylabel ("Trainingszeit (Minuten)")
    ax_time .legend (loc ="upper left")
    for x ,y in zip (x_vals ,df ["training_time_seconds"]/60.0 ):
        ax_time .annotate (f"{y :.1f} min",(x ,y ),textcoords ="offset points",xytext =(0 ,7 ),ha ='center',fontsize =9 )

    plt .tight_layout ()
    plot_file =os .path .join (PLOTS_DIR ,"sft_scaling_comprehensive_curves.png")
    plt .savefig (plot_file ,dpi =300 )
    print (f"6-Panel Scaling Plot erfolgreich gespeichert unter: {plot_file }")
    plt .show ()
else :
    print ("Keine Daten fuer Plots vorhanden.")


## 3. Detaillierter Vergleich: SFT-Skalierungsplateau vs. DPO Alignment

In [ ]:
if not df .empty :
    fig ,ax =plt .subplots (figsize =(10 ,6 ))

    models =[f"SFT {int (f *100 )}% (N={n })"for f ,n in zip (df ["train_fraction"],df ["num_train_pairs"])]+["DPO Mean (N=480)"]
    simp_scores =list (df ["r_style_mean"])+[0.6938 ]
    comp_scores =list (df ["composite_reward_mean"])+[0.7767 ]

    x =np .arange (len (models ))
    width =0.35 

    rects1 =ax .bar (x -width /2 ,simp_scores ,width ,label ='Simplicity ($R_{style}$)',color ='#2ca02c',alpha =0.85 )
    rects2 =ax .bar (x +width /2 ,comp_scores ,width ,label ='Composite Reward',color ='#1f77b4',alpha =0.85 )

    ax .set_ylabel ('Score [0.0 - 1.0]')
    ax .set_title ('Empirischer Vergleich: SFT-Datenvermehrung vs. DPO Preference Alignment',fontweight ='bold')
    ax .set_xticks (x )
    ax .set_xticklabels (models ,rotation =15 ,ha ='right')
    ax .set_ylim (0.0 ,0.9 )
    ax .legend (loc ='upper left')

    # Werte über Balken
    for rect in rects1 :
        height =rect .get_height ()
        ax .annotate (f'{height :.3f}',xy =(rect .get_x ()+rect .get_width ()/2 ,height ),
        xytext =(0 ,3 ),textcoords ="offset points",ha ='center',va ='bottom',fontsize =9 ,fontweight ='bold')
    for rect in rects2 :
        height =rect .get_height ()
        ax .annotate (f'{height :.3f}',xy =(rect .get_x ()+rect .get_width ()/2 ,height ),
        xytext =(0 ,3 ),textcoords ="offset points",ha ='center',va ='bottom',fontsize =9 )

    plt .tight_layout ()
    dpo_comp_file =os .path .join (PLOTS_DIR ,"sft_vs_dpo_comparison_bar.png")
    plt .savefig (dpo_comp_file ,dpi =300 )
    print (f"Vergleichs-Plot gespeichert unter: {dpo_comp_file }")
    plt .show ()


## 4. Qualitativer Side-by-Side Modellvergleich (Beispielsätze)
Untersucht die linguistische Entwicklung der Übersetzungen von $10\%$ bis $100\%$ Trainingsdaten.

In [ ]:
detail_files =sorted (glob .glob (os .path .join (RESULTS_DIR ,"*_details.csv")))
if detail_files :
    dfs_det =[pd .read_csv (f )for f in detail_files ]
    df_all_det =pd .concat (dfs_det ,ignore_index =True )

    sample_as =df_all_det ["as_text"].unique ()[:2 ]
    for idx ,as_query in enumerate (sample_as ,1 ):
        sub_df =df_all_det [df_all_det ["as_text"]==as_query ].sort_values (by ="train_fraction")
        ls_ref =sub_df ["ls_ref_text"].iloc [0 ]

        print (f"\n{'#'*90 }")
        print (f"BEISPIEL {idx }:")
        print (f"[AS Quelle] : {as_query [:160 ]}...")
        print (f"[LS Referenz]: {ls_ref [:160 ]}...")
        print (f"{'-'*90 }")

        for _ ,row in sub_df .iterrows ():
            f_pct =int (row ['train_fraction']*100 )
            print (f"[{row ['experiment_name']:20s} ({f_pct :3d}%)] (Simp: {row ['r_style']:.3f} | Sem: {row ['r_sem_as']:.3f} | Tokens: {row ['gen_tokens']}):")
            print (f"  {row ['generated_text']}\n")
else :
    print ("Keine Detail-CSV Dateien verfuegbar.")
